In [1]:
import sys
sys.path.append(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq")

from TCC.utils.constantes import *
import matplotlib.pyplot as plt

## SANTIMENT SOCIAL METRICS FOR A SELECTED GROUP OF WORDS
- O Google Trends não fornece o número absoluto de buscas, mas sim um índice de interesse normalizado.O valor $100$ representa o pico de popularidade do termo no período e local selecionados. Um valor de $50$ significa que o termo teve metade da popularidade daquele pico.

- Fonte1: https://academy.santiment.net/metrics/social-dominance/
- Fonte2: https://academy.santiment.net/metrics/social-volume/#definition

### HIPÓTESES
- Hipótese (Era Varejo)Correlação Positiva Direta com $BTC.D$. Pré-2020: O varejo era o motor. Se Social_Dom_Spread subia (mais papo de BTC), o $BTC.D$ subia. Se o Spread caía (mais papo de Alts), o $BTC.D$ desabava (Altseason). O efeito era imediato e linear.Hipótese (Era Institucional)Sinal "Contrarian" (Topo de Euforia). 

- Pós-2020: Vendas ou compras institucionais não são baseados em euforia nas redes sociais, logo espera-se que a dominancia social nao seja um fator muito decisivo na correlacao da domianncia do bitcoin e vice versa.

### TRATAMENTO
- O Spread Negativo (Média -5.87): Como somou-se o volume de todas as Altcoins (ETH + SOL + ADA + "Altcoin Season"...), o volume agregado delas será sempre maior que o do Bitcoin sozinho. O que importa para o modelo não é o sinal negativo, mas a variação (se for de -5 para -3, o BTC ganhou força relativa).

O Z-Score (Total Market Noise):Isso permite que o modelo saiba a diferença entre "Mercado em Silêncio" (Z < -1) e "Euforia Generalizada" (Z > 2). O Bitcoin se comporta de forma diferente nesses dois regimes.

Tratamento de Zeros: O uso de np.log1p (log de x+1) é a prática correta para evitar erros matemáticos em dias de volume zero.

In [13]:
df_social = pd.read_csv(rf"raw/2012_btc_altcoin_social_metrics.csv")
df_social['Data_UTC'] = pd.to_datetime(df_social['Date'], utc=True).dt.strftime("%Y-%m-%d")

df_social = df_social.rename(columns={'BTC / USD':'btc_price',
                                  'Social Volume (btc OR bitcoin)':'btc_social_volume',
                                  'Social Dominance (btc OR bitcoin)':'btc_social_dominance',
                                  'Social Volume (altcoin OR altcoins OR "altcoin season" OR "altcoin pump" OR "altcoin rally")':'alt_social_volume',
                                  'Social Dominance (altcoin OR altcoins OR "altcoin season" OR "altcoin pump" OR "altcoin rally")':'alt_social_dominance',
                                  'Social Volume (ethereum OR eth OR solana OR sol OR cardano OR ada OR polkadot OR dot OR ripple OR xrp OR dogecoin OR doge OR memecoin OR shiba)':'alt_social_volume_2',
                                  'Social Dominance (ethereum OR eth OR solana OR sol OR cardano OR ada OR polkadot OR dot OR ripple OR xrp OR dogecoin OR doge OR memecoin OR shiba)':'alt_social_dominance_2',
                                  })

df_social['Data_UTC'] = pd.to_datetime(df_social['Data_UTC'], utc=True,).dt.strftime("%Y-%m-%d")

df_social

df_social_tratado = (
    df_periodo
        .merge(df_social, how='left', on='Data_UTC')
        .sort_values('Data_UTC')
        
        # 1. Transformação Logarítmica (CRUCIAL para dados de contagem)
        # np.log1p evita erro de log(0) se o volume for zero.
        .assign(btc_log_vol = lambda df: np.log1p(df['btc_social_volume']))
        .assign(alt_log_vol_1 = lambda df: np.log1p(df['alt_social_volume']))
        .assign(alt_log_vol_2 = lambda df: np.log1p(df['alt_social_volume_2']))
        .assign(alt_log_dom_1 = lambda df: np.log1p(df['alt_social_dominance']))
        .assign(alt_log_dom_2 = lambda df: np.log1p(df['alt_social_dominance_2']))

        # 2. Social Volume e Dominance (Soma dos Altcoins)
        .assign(alt_total_log_vol = lambda df: df['alt_log_vol_1'] + df['alt_log_vol_2'])
        .assign(alt_total_log_dom = lambda df: df['alt_log_dom_1'] + df['alt_log_dom_2'])
        
        # 3. O Spread de Intensidade (Feature Principal)
        # Positivo: O barulho do Bitcoin é maior (Logaritmicamente)
        # Negativo: As Altcoins estão gritando mais alto
        .assign(social_vol_log_spread = lambda df: df['btc_log_vol'] - df['alt_total_log_vol'])
        
        # 43. Aceleração do Hype (Derivada)
        # O barulho está aumentando ou diminuindo em relação a ontem?
        .assign(social_momentum = lambda df: df['social_vol_log_spread'].diff())

        # 4. (Opcional) Z-Score Total do Mercado
        # O mercado está eufórico (muito volume geral) ou apático?
        .assign(total_market_noise_z = lambda df: 
                ((df['btc_log_vol'] + df['alt_total_log_vol']) - (df['btc_log_vol'] + df['alt_total_log_vol']).rolling(30).mean()) /
                (df['btc_log_vol'] + df['alt_total_log_vol']).rolling(30).std()
        )
        .fillna(0)

        [['Data_UTC', 'social_vol_log_spread', 'social_momentum', 'total_market_noise_z']]
)

df_social_tratado

print_dataframe_info(df_social_tratado, "Social Metrics ")


Informações do Social Metrics 

DESCRIBE:
       social_vol_log_spread  social_momentum  total_market_noise_z
count            3136.000000      3136.000000           3136.000000
mean               -5.872511        -0.000649             -0.007029
std                 0.926236         0.276040              1.153039
min                -8.477794        -2.047912             -3.904291
25%                -6.635864        -0.156165             -0.869690
50%                -5.739116         0.005944             -0.037984
75%                -5.211571         0.156005              0.797859
max                -3.038792         2.681576              4.073126

--------------------------------------------------

INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3136 entries, 0 to 3135
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Data_UTC               3136 non-null   object 
 1   social_vol_log_spre